In [ ]:
from google.colab import drive

# force_remount=True로 강제 재마운트
drive.mount('/content/drive/', force_remount=True)

Mounted at /content/drive/


In [ ]:
pip install chromatic_tda

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.6/144.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.0/189.0 kB 11.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.4.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.0 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.4.0 which is incompatible.
tensorf

In [ ]:
pip install gudhi

  Using cached gudhi-3.11.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (1.6 kB)
Using cached gudhi-3.11.0-cp312-cp312-manylinux_2_28_x86_64.whl (4.2 MB)


In [ ]:
pip install persim

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 1.9 MB/s eta 0:00:00
  Created wheel for hopcroftkarp: filename=hopcroftkarp-1.2.5-py2.py3-none-any.whl size=18104 sha256=cfb31a48d40ed4a03fa10ea72374c46d3148a77dc552b9128022031fc8493aae
  Stored in directory: /root/.cache/pip/wheels/2a/fd/fe/f4b8fd82894e1d9e04040ef41dc5ae6eb7a8e9b0ef5a9402fe
Successfully built hopcroftkarp


In [ ]:
import chromatic_tda as chro
import matplotlib.pyplot as plt
import numpy as np
from gudhi import RipsComplex
from gudhi.representations import PersistenceImage

from persim import PersistenceImager
import persim.images_weights as weights

In [ ]:
def convert_into_diagram(diagram):
    """
    Converts kernel diagram dict → {dim: ndarray} and removes infinite deaths.
    """
    diagrams = {}
    for dim, pairs in diagram.items():
        filtered = [(float(b), float(d)) for (b, d) in pairs if np.isfinite(d)]
        diagrams[dim] = np.array(filtered)
    return diagrams

def convert_six_pack_to_diagram(six_pack):
    packs = {}

    for key, dgm in six_pack.items():
        pack = convert_into_diagram(dgm)
        packs[key] = pack

    return packs

In [ ]:
def compute_six_pack_diagrams(points, labels, max_edge = 10):
    chro_alpha = chro.ChromaticAlphaComplex(points, labels, max_alpha = max_edge)

    simplicial_complex = chro_alpha.get_simplicial_complex(
        sub_complex='0',
        full_complex='all',
        relative= None
    )

    six_pack = simplicial_complex.bars_six_pack()
    six_pack_dgms = convert_six_pack_to_diagram(six_pack)

    return six_pack_dgms

In [ ]:
def compute_PIs(barcodes, max_eps=10, px_res=0.1, sigma=0.05, normalization=False):
    # =========================================================================
    # 키가 존재하지 않는 경우 빈 배열로 초기화 (KeyError 방지)
    # =========================================================================
    if 0 not in barcodes:
        barcodes[0] = np.zeros((0, 2))
    if 1 not in barcodes:
        barcodes[1] = np.zeros((0, 2))

    for key in barcodes.keys():
        if len(barcodes[key]) == 0:
            barcodes[key] = np.zeros((0, 2))

    vector = {}

    # =========================================================================
    # H0 Persistence Image
    # 논문: birth_range=(0, 1), pers_range=(0, max_eps), skew=False
    # =========================================================================
    pers_imager_h0 = PersistenceImager()
    pers_imager_h0.pixel_size = px_res
    pers_imager_h0.birth_range = (0, 0.01)
    pers_imager_h0.pers_range = (0, max_eps)
    pers_imager_h0.weight = weights.persistence
    pers_imager_h0.weight_params = {'n': 1}
    pers_imager_h0.kernel_params = {'sigma': [[sigma, 0], [0, sigma]]}

    bars_h0 = np.array(barcodes.get(0, np.zeros((0, 2))))
    if len(bars_h0) > 0:
        # H0는 skew=False (이미 birth-persistence 형태로 간주)
        img_h0 = pers_imager_h0.transform(bars_h0, skew=True)
    else:
        img_h0 = np.zeros((int(1/px_res), int(max_eps/px_res)))

    # 논문: np.mean(img, axis=0) - birth 축을 따라 평균
    img0_1d = np.mean(img_h0, axis=0)

    # =========================================================================
    # H1 Persistence Image
    # 논문: birth_range=(0, max_eps), pers_range=(0, max_eps/2), skew=True
    # =========================================================================
    pers_imager_h1 = PersistenceImager()
    pers_imager_h1.pixel_size = px_res
    pers_imager_h1.birth_range = (0, max_eps)
    pers_imager_h1.pers_range = (0, max_eps / 2)
    pers_imager_h1.weight = weights.persistence
    pers_imager_h1.weight_params = {'n': 1}
    pers_imager_h1.kernel_params = {'sigma': [[sigma, 0], [0, sigma]]}

    bars_h1 = np.array(barcodes.get(1, np.zeros((0, 2))))
    if len(bars_h1) > 0:
        # H1은 skew=True ((birth, death) → (birth, persistence) 변환)
        img_h1 = pers_imager_h1.transform(bars_h1, skew=True)
    else:
        img_h1 = np.zeros((int(max_eps/px_res), int((max_eps/2)/px_res)))

    # =========================================================================
    # Normalization & Output
    # =========================================================================
    if normalization:
        if np.max(img0_1d) > 0:
            vector[0] = img0_1d / np.max(img0_1d)
        else:
            vector[0] = img0_1d

        if np.max(img_h1) > 0:
            vector[1] = img_h1.flatten() / np.max(img_h1)
        else:
            vector[1] = img_h1.flatten()
    else:
        vector[0] = img0_1d
        vector[1] = img_h1.flatten()

    return vector

def visualize_PIs(PIs, max_eps=10, px_res=0.1):
    import matplotlib.pyplot as plt

    # H0: shape (100,) - 1D image
    # 논문에서 mean(axis=0)으로 1D가 되었으므로 그대로 plot
    h0_img = PIs[0]
    h0_length = int(max_eps / px_res)  # 100

    # H1: shape (5000,) → reshape to (100, 50)
    h1_birth_bins = int(max_eps / px_res)  # 100
    h1_pers_bins = int((max_eps / 2) / px_res)  # 50
    h1_img = PIs[1].reshape((h1_birth_bins, h1_pers_bins))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # H0 (1D plot)
    axes[0].plot(h0_img, 'b-', linewidth=1.5)
    axes[0].fill_between(range(len(h0_img)), h0_img, alpha=0.3)
    axes[0].set_title(f'H0 Persistence Image (1D)\nLength: {h0_length}')
    axes[0].set_xlabel('Persistence (Bin Index)')
    axes[0].set_ylabel('Intensity')
    axes[0].set_xlim([0, h0_length])
    axes[0].grid(True, alpha=0.3)

    # H1 (2D image)
    im = axes[1].imshow(h1_img.T, cmap='hot', origin='lower', aspect='auto')
    axes[1].set_title(f'H1 Persistence Image (2D)\nResolution: ({h1_birth_bins}, {h1_pers_bins})')
    axes[1].set_xlabel('Birth (Bin Index)')
    axes[1].set_ylabel('Persistence (Bin Index)')
    plt.colorbar(im, ax=axes[1], label='Intensity')

    plt.tight_layout()
    plt.show()

In [ ]:
import os
BASE_DIR = "/content/drive/MyDrive/URP/1224_Vectors"
BASE_DIR1 = "/content/drive/MyDrive/URP"
OUT_DIR_1  = os.path.join(BASE_DIR, "Sixpack_Chroma")
#OUT_DIR_2 = os.path.join(BASE_DIR, "Sixpack_Chroma")
os.makedirs(OUT_DIR_1, exist_ok=True)

A_VALS = [0.0, 0.01, 0.05, 0.09, 0.13, 0.17, 0.21, 0.25]
PARAM_LIST = [(x1, x2, x3) for x1 in A_VALS for x2 in A_VALS for x3 in A_VALS]

In [ ]:
import time
for idx in range(60, 61):
    print(f"[{idx}] 실행 시작")
    start_time = time.time()   # 시작 시간 기록
    params = PARAM_LIST[idx - 1]
    folder = os.path.join(BASE_DIR1, f"ParamSweep_{idx}_Output")
    pos_file_path   = os.path.join(folder, f"Pos_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}.dat")
    types_file_path = os.path.join(folder, f"Types_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}.dat")
    save_path_1  = os.path.join(OUT_DIR_1, f"Sixpack_Chroma_{idx}.npz")
    #save_path_2  = os.path.join(OUT_DIR_2, f"Sixpack_Chroma_{idx}.npz")
    types = np.loadtxt(types_file_path, dtype=int)
    positions = np.loadtxt(pos_file_path, delimiter=',')
    A = positions[types == 1]
    B = positions[types == 2]
    points = np.array(np.concatenate([A, B], axis = 0))
    labels = np.array(np.concatenate([np.zeros(len(A)), np.ones(len(B))]))

    six_pack_A_to_B = compute_six_pack_diagrams(points, labels, max_edge = 10)
    six_pack_B_to_A = compute_six_pack_diagrams(points, 1-labels, max_edge = 10)
    PI_six_pack_A_to_B = {}
    PI_six_pack_B_to_A = {}

    for key in six_pack_A_to_B.keys():
        PI = compute_PIs(six_pack_A_to_B[key], normalization = False)
        PI_six_pack_A_to_B[key] = PI


    for key in six_pack_B_to_A.keys():
        PI = compute_PIs(six_pack_B_to_A[key], normalization = False)
        PI_six_pack_B_to_A[key] = PI
    np.savez_compressed(save_path_1,
                            PI_six_pack_A_to_B,
                            PI_six_pack_B_to_A)
    end_time = time.time()     # 종료 시간 기록
    elapsed = end_time - start_time
    print(f"[{idx}] 실행 완료 (걸린 시간: {elapsed:.2f} 초)")


[60] 실행 시작
[60] 실행 완료 (걸린 시간: 5.87 초)
